# Pooling across states, scored the way CDC scores

Notebook 04 showed that CDC's FluSight ensemble beats our SARIMA mainly through its **range**, and that averaging models which see the same national series barely helps. The ensemble's members see *different* data.

This notebook does two things:
1. **Scores every forecast with WIS**, the weighted interval score CDC uses to rank FluSight teams.
2. **Builds a model pooled across all 53 locations** (states, DC, Puerto Rico and the US), following the published design of UMass's `flusion` AR component, and compares it with SARIMA and the professionals.

**One design, run once.** The pooled model's choices (fourth-root rates, 6 weeks of history, coefficients shared across locations, range from past errors) were fixed in advance from UMass's published description and evaluated once on these dates, with no tuning. Every later change is reported separately as a declared experiment.

## 0. Setup

Same data and SARIMA as notebooks 03 and 04, on the **55 flu-season forecast dates** that FluSight covers.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import io, urllib.request
import pandas as pd, numpy as np
from scipy.stats import norm
from statsmodels.tsa.statespace.sarimax import SARIMAX

HUB = 'https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/main/'
raw = pd.read_csv(HUB + 'target-data/target-hospital-admissions.csv', parse_dates=['date'])
y = raw[raw.location == 'US'].set_index('date')['value'].sort_index().asfreq('W-SAT')

# FluSight ensemble and baseline: all 23 percentiles, US, weekly admissions
BASE = HUB + 'model-output/{m}/{d}-{m}.csv'
parts = []
for m in ['FluSight-ensemble', 'FluSight-baseline']:
    for r in pd.date_range('2024-08-03', '2026-06-06', freq='W-SAT'):
        try:
            data = urllib.request.urlopen(BASE.format(m=m, d=r.strftime('%Y-%m-%d'))).read()
        except Exception:
            continue
        f = pd.read_csv(io.BytesIO(data), dtype={'location': str, 'output_type_id': str})
        f = f[(f['location'] == 'US') & (f['target'] == 'wk inc flu hosp') & (f['output_type'] == 'quantile')]
        f['model'] = m
        parts.append(f)
fs = pd.concat(parts)
fs['cutoff']  = pd.to_datetime(fs['reference_date']) - pd.Timedelta(weeks=1)   # = our cutoff
fs['horizon'] = fs['horizon'] + 1                                              # = our horizon
fs = fs[fs['horizon'].between(1, 4)].rename(columns={'value': 'value'})
fs['tau'] = fs['output_type_id'].astype(float)

cutoffs = sorted(fs['cutoff'].unique())
TAUS = sorted(fs['tau'].unique())
truth = pd.DataFrame([{'cutoff': c, 'horizon': h, 'actual': y[c + pd.Timedelta(weeks=h)]}
                      for c in cutoffs for h in range(1, 5)])
print(len(cutoffs), 'forecast dates |', len(TAUS), 'percentiles per forecast')

55 forecast dates | 23 percentiles per forecast


## 1. WIS: one score for the whole range

MAE judges only the middle line. **WIS judges every percentile**, rewarding ranges that are both narrow and honest.

For one percentile τ (e.g. q90, τ = 0.9):
- reality **above** it: penalty = τ × gap (rare, so expensive)
- reality **below** it: penalty = (1 − τ) × gap (expected, so cheap)

For the median both prices are 0.5, so it reduces to absolute error. **WIS = 2 × the average penalty over the 23 percentiles.** It's in admissions; lower is better. CDC also reports **relative WIS** = WIS ÷ FluSight-baseline's WIS: below 1 beats the baseline.

In [2]:
def pinball(actual, q_value, tau):
    gap = actual - q_value
    return np.where(gap > 0, tau * gap, (tau - 1) * gap)

def wis(quantiles):
    """quantiles: rows of (model, cutoff, horizon, tau, value). Returns one WIS per forecast."""
    q = quantiles.merge(truth, on=['cutoff', 'horizon'])
    q['loss'] = pinball(q['actual'], q['value'], q['tau'])
    return (q.groupby(['model', 'cutoff', 'horizon'])['loss'].mean() * 2).rename('wis').reset_index()

wis_fs = wis(fs[['model', 'cutoff', 'horizon', 'tau', 'value']])
print(wis_fs.pivot_table(index='horizon', columns='model', values='wis').round(0))

model    FluSight-baseline  FluSight-ensemble
horizon                                      
1.0                 2646.0             1857.0
2.0                 4502.0             2910.0
3.0                 6305.0             4007.0
4.0                 7902.0             5067.0


The ensemble's WIS (5,067 at 4 weeks) is **lower than its MAE** (7,104): when reality lands inside a well-placed range, the outer percentiles are on their cheap side, so the average comes out below the middle line's error. A badly placed range gets no such discount.

## 2. SARIMA with 23 percentiles

SARIMA's forecast is a bell curve: a middle (`predicted_mean`) and a spread (`se_mean`). Percentile τ = middle + z(τ) × spread, clipped at 0. The middle line used for MAE is floored at the lowest week seen so far, as in notebook 03.

In [3]:
def fit_sarima(series):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return SARIMAX(series, order=(1,1,1), seasonal_order=(1,0,0,52),
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

sq, sp = [], []
for c in cutoffs:
    f = fit_sarima(y[:c]).get_forecast(4)
    mu, se = f.predicted_mean.values, f.se_mean.values
    for h in range(4):
        sp.append({'cutoff': c, 'horizon': h + 1, 'SARIMA': max(mu[h], y[:c].min())})
        for t in TAUS:
            sq.append({'model': 'SARIMA', 'cutoff': c, 'horizon': h + 1, 'tau': t,
                       'value': max(mu[h] + norm.ppf(t) * se[h], 0)})
sarima_q, sarima_point = pd.DataFrame(sq), pd.DataFrame(sp)
wis_sarima = wis(sarima_q)
print('done')

done


## 3. The pooled model

**Why pool?** SARIMA learns from one curve, about four seasons. The FluSight file has **53 curves**. Each state's season is another example of how flu epidemics rise and turn. Pooled, the model learns from ~200 season-curves.

**Same scale for every location:**
1. **`weekly_rate`** (admissions per 100,000 people) instead of counts, so big and small states are comparable.
2. **Fourth root**, which compresses large values (like log, but gentler, so less compounding overshoot). UMass uses this transform.

**The model:** one linear regression per horizon, **shared by all locations**.
- inputs: the last 6 weekly changes and the current level (fourth-root scale)
- target: the change from now to *h* weeks ahead

To forecast the nation, apply the shared coefficients to the US's own recent changes and undo the transform.

**Range:** the forecast plus percentiles of the regression's past errors, across all locations: ~7,000 past mistakes at every forecast date (notebook 03's calibration had ~50).

**No peeking:** at each forecast date, train only on examples whose target week had already happened.

In [4]:
rate = raw.pivot_table(index='date', columns='location_name', values='weekly_rate').sort_index().asfreq('W-SAT')
z = rate ** 0.25                                    # fourth root of admissions per 100k
us_scale = (y / rate['US']).median()                # US rate -> admissions

K = 6
feat = {f'chg{k}': z.diff().shift(k) for k in range(K)}   # chg0 = this week's change, chg1 = last week's, ...
feat['level'] = z
tables = []
for h in range(1, 5):
    tb = pd.concat({name: f.stack() for name, f in feat.items()}, axis=1)   # one row per (week, location)
    tb['target'] = (z.shift(-h) - z).stack()                                 # change h weeks later
    tb['horizon'] = h
    tables.append(tb)
train = pd.concat(tables).reset_index().rename(columns={'level_1': 'location_name'})
train['target_date'] = train['date'] + pd.to_timedelta(train['horizon'], unit='W')
cols = [f'chg{k}' for k in range(K)] + ['level']
print(z.shape, 'weeks x locations |', train.shape, 'training rows x columns')

(244, 53) weeks x locations | (51728, 12) training rows x columns


In [5]:
def pooled_forecasts(train, label='pooled'):
    points, quants = [], []
    for c in cutoffs:
        for h in range(1, 5):
            tr = train[(train['horizon'] == h) & (train['target_date'] <= c)].dropna()   # no peeking
            X = np.c_[tr[cols].values, np.ones(len(tr))]
            beta = np.linalg.lstsq(X, tr['target'].values, rcond=None)[0]                # linear regression
            errors = tr['target'].values - X @ beta                                       # past mistakes
            now = train[(train['horizon'] == h) & (train['location_name'] == 'US') & (train['date'] == c)]
            zhat = now['level'].iloc[0] + np.r_[now[cols].values[0], 1.0] @ beta
            points.append({'cutoff': c, 'horizon': h, label: max(zhat, 0) ** 4 * us_scale})
            for t in TAUS:
                quants.append({'model': label, 'cutoff': c, 'horizon': h, 'tau': t,
                               'value': max(zhat + np.quantile(errors, t), 0) ** 4 * us_scale})
    return pd.DataFrame(points), pd.DataFrame(quants)

pooled_point, pooled_q = pooled_forecasts(train)
wis_pooled = wis(pooled_q)
print('done')

done


## 4. Results

### Accuracy of the middle line (MAE)

In [6]:
med = fs[fs['tau'] == 0.5].pivot_table(index=['cutoff', 'horizon'], columns='model', values='value').reset_index()
pt = med.merge(sarima_point, on=['cutoff', 'horizon']).merge(pooled_point, on=['cutoff', 'horizon']).merge(truth, on=['cutoff', 'horizon'])
models = ['FluSight-ensemble', 'pooled', 'SARIMA', 'FluSight-baseline']
mae = pd.DataFrame({mm: (pt[mm] - pt['actual']).abs().groupby(pt['horizon']).mean() for mm in models})
print(mae.round(0))

         FluSight-ensemble  pooled  SARIMA  FluSight-baseline
horizon                                                      
1.0                 2652.0  3826.0  2550.0             3328.0
2.0                 4144.0  5368.0  5715.0             5750.0
3.0                 5638.0  6785.0  8098.0             8104.0
4.0                 7104.0  7859.0  9764.0            10023.0


### Relative WIS (÷ FluSight-baseline; below 1 is better)

In [7]:
w = pd.concat([wis_fs, wis_sarima, wis_pooled]).pivot_table(index='horizon', columns='model', values='wis')
print((w[models].div(w['FluSight-baseline'], axis=0)).round(2))

model    FluSight-ensemble  pooled  SARIMA  FluSight-baseline
horizon                                                      
1.0                   0.70    0.97    0.77                1.0
2.0                   0.65    0.81    1.02                1.0
3.0                   0.64    0.72    1.03                1.0
4.0                   0.64    0.67    0.99                1.0


### Are the 80% ranges honest?

In [8]:
allq = pd.concat([fs[['model', 'cutoff', 'horizon', 'tau', 'value']], sarima_q, pooled_q])
r80 = (allq[allq['tau'].isin([0.1, 0.9])]
         .pivot_table(index=['model', 'cutoff', 'horizon'], columns='tau', values='value').reset_index()
         .merge(truth, on=['cutoff', 'horizon']))
r80['inside'] = (r80['actual'] >= r80[0.1]) & (r80['actual'] <= r80[0.9])
r80['above']  = r80['actual'] > r80[0.9]
print(r80.groupby('model')[['inside', 'above']].mean().loc[models].round(2), '\n(targets: inside 0.80, above 0.10)')

                   inside  above
model                           
FluSight-ensemble    0.77   0.20
pooled               0.84   0.14
SARIMA               0.61   0.20
FluSight-baseline    0.60   0.24 
(targets: inside 0.80, above 0.10)


### The four turns (4 weeks ahead)

In [9]:
turns = pd.to_datetime(['2024-12-28', '2025-02-01', '2025-12-20', '2025-12-27'])
pt[(pt['horizon'] == 4) & pt['cutoff'].isin(turns)][['cutoff', 'actual', 'SARIMA', 'pooled', 'FluSight-ensemble']].round(0)

,cutoff,actual,SARIMA,pooled,FluSight-ensemble
27,2024-12-28,41399.0,63775.0,21237.0,23735.0
43,2025-02-01,31905.0,61570.0,25942.0,33810.0
131,2025-12-20,19731.0,36965.0,18712.0,30159.0
135,2025-12-27,16816.0,54715.0,32688.0,39965.0


### Staffing (4 weeks ahead, understaffing costs 3, overstaffing 1)

In [10]:
def cost(actual, capacity, ratio=3):
    return np.maximum(capacity - actual, 0).sum() + ratio * np.maximum(actual - capacity, 0).sum()

h4 = (allq[(allq['horizon'] == 4) & allq['tau'].isin([0.5, 0.75, 0.9])]
        .pivot_table(index=['cutoff'], columns=['model', 'tau'], values='value'))
h4.columns = [f'{mm} q{int(t * 100)}' for mm, t in h4.columns]
h4 = h4.reset_index().merge(truth[truth['horizon'] == 4], on='cutoff')
plans = [col for col in h4.columns if ' q' in col]
out = pd.DataFrame({'cost': [cost(h4['actual'], h4[p]) for p in plans],
                    'weeks short': [(h4['actual'] > h4[p]).mean() for p in plans]}, index=plans)
print(out.sort_values('cost').round(2))

                             cost  weeks short
FluSight-ensemble q90   807203.00         0.24
FluSight-ensemble q75   842063.00         0.31
pooled q75              932498.41         0.33
pooled q90              932826.79         0.18
FluSight-ensemble q50   978002.00         0.64
SARIMA q50              998806.31         0.40
SARIMA q75             1009648.49         0.25
SARIMA q90             1083829.15         0.20
FluSight-baseline q50  1153056.00         0.31
pooled q50             1154152.44         0.73
FluSight-baseline q75  1172577.00         0.27
FluSight-baseline q90  1214996.00         0.25


**What it shows.**
- **Accuracy:** the pooled model cuts SARIMA's 4-week MAE by ~20% (9,764 → 7,859), closing ~70% of the gap to the ensemble (7,104). It is worse at 1 week, where SARIMA's US-specific momentum matters most.
- **WIS:** at 4 weeks the pooled model scores **0.67**, within 0.03 of the ensemble (0.64); SARIMA scores 0.99. The best combination is SARIMA at 1 week and pooled at 2-4 weeks: about 0.77 / 0.81 / 0.72 / 0.67.
- **Ranges:** the pooled 80% range is the most honest of all: **84% inside, 14% above q90**, vs the ensemble's 77% / 20%. Calibration from past errors works once it has ~7,000 errors to learn from.
- **Turns:** pooling beat the ensemble at both 2025-26 turns and more than halved SARIMA's worst overshoot. It doesn't predict turns; it learned from ~200 season-curves that fast climbs rarely continue at full speed for four weeks. The flip side: it undershot the double-peak season's continued climb (28 Dec 2024), as the ensemble did.
- **Staffing:** the best pooled plan (~932k) is ~7% cheaper than SARIMA's best (~998k), but ~15% dearer than the ensemble's (~807k). The pooled median is short ~73% of weeks: its caution makes it under-forecast the many climbing weeks, and at a 3:1 cost shortfalls dominate. WIS prices mistakes more symmetrically, which is why it rates the pooled model more highly than staffing does. The metric should match the decision.

## 5. Summary

| 4 weeks ahead, 55 flu-season dates | MAE | Relative WIS | 80% range inside / above | Best staffing cost |
|---|---|---|---|---|
| FluSight ensemble | 7,104 | 0.64 | 77% / 20% | 807k |
| **Pooled (this project)** | 7,859 | 0.67 | **84% / 14%** | 932k |
| SARIMA | 9,764 | 0.99 | 61% / 20% | 998k |
| FluSight baseline | 10,024 | 1.00 | 60% / 24% | 1,153k |

**Caveats.** 55 dates over two flu seasons. FluSight forecast from real-time data, this backtest uses revised data, which favours our models. One pooled design, chosen in advance and run once.

**Next (declared experiment).** The pooled median is too low in flu season. One pre-declared change will be tested: **train the pooled model on flu-season weeks only**. Success = weeks short for the median closer to 50% and lower staffing cost, without a worse relative WIS. The result will be reported whether or not it helps.